# 01_day1_generate_code.ipynb
==================================================

## Importing and Setup


In [1]:
# Install libraries from inside the notebook
%pip install polars pandas numpy faker plotly scikit-learn xgboost streamlit --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Importing required libraries for data handling and random data generation
import polars as pl
import pandas as pd
import numpy as np
import random
from pathlib import Path
from faker import Faker

# Setup Faker to generate Indian names and cities
faker = Faker("en_IN")

# Define the data folder path (already created)
# This insures CSV are stored in the correct folder automatically
data_folder = Path(
    "/Users/Mudit PC/OneDrive/Desktop/Data analytics projects/Indian_Beauty_Analytics/data/"
)

## Generate Product Data


In [3]:
# Define a list of products with product_id, name, category and price
products = [
    {"product_id": f"p{i}", "product_name": name, "category": cat, "price": price}
    for i, (name, cat, price) in enumerate(
        [
            ("Aloe Vera Gel", "Skincare", 299),
            ("Vitamin C Serum", "Skincare", 499),
            ("Herbal Shampoo", "Haircare", 199),
            ("Lip Balm", "Makeup", 149),
            ("Face Wash", "Skincare", 399),
            ("Sunscreen SPF50", "Skincare", 599),
            ("Hair Oil", "Haircare", 249),
            ("Body Lotion", "Skincare", 349),
            ("Nail Polish", "Makeup", 199),
            ("Face Pack", "Skincare", 299)
        ]
    )
]

# Convert the product list into a polars DataFrame
products_df = pl.DataFrame(products)

# Save the products DataFrame to csv inside the data folder
products_df.write_csv(data_folder / "products.csv")

# Display the first few rows of the products DataFrame
products_df.head()

product_id,product_name,category,price
str,str,str,i64
"""p0""","""Aloe Vera Gel""","""Skincare""",299
"""p1""","""Vitamin C Serum""","""Skincare""",499
"""p2""","""Herbal Shampoo""","""Haircare""",199
"""p3""","""Lip Balm""","""Makeup""",149
"""p4""","""Face Wash""","""Skincare""",399


## Generate Customer Data


In [4]:
# Set the number of customers to generate
num_customers = 1000

# Initialize an emply list to store customer dictionaries
customers = []

# Loop to generate each customer
for i in range(num_customers):
    customers.append({
        "customer_id": f"c{i}",  # Unique Customer ID
        "name": faker.name(),    # Random Indian Customer Name
        "city": faker.city(),   # Random Indian City
        "age": random.randint(18, 55),  # Random Age between 18 and 55
        "gender": random.choice(["Male", "Female"]),  # Random Gender
    }
    )

# Convert the customer list into a polars DataFrame
customers_df = pl.DataFrame(customers)

# Save the customers DataFrame to csv inside the data folder
customers_df.write_csv(data_folder / "customers.csv")

# Display the first few rows of the customers DataFrame
customers_df.head()

customer_id,name,city,age,gender
str,str,str,i64,str
"""c0""","""Brijesh Munshi""","""Bikaner""",26,"""Male"""
"""c1""","""Tanveer Kade""","""Bhubaneswar""",43,"""Male"""
"""c2""","""Hamsini Tripathi""","""Anantapuram""",42,"""Male"""
"""c3""","""Ekantika Chada""","""Sasaram""",27,"""Female"""
"""c4""","""Jeet Shere""","""Gaya""",32,"""Male"""


## Generate Transactions


In [ ]:
# Set the number of transactions to generate
num_transactions = 50000

# Initialize an empty list to store transaction dictionaries
transactions = []

# Define festival periods with start and end dates
festivals = {
    "Diwali": ("2024-10-10", "2024-11-10"),
    "Holi": ("2024-03-01", "2024-03-31"),
    "Summer": ("2024-05-01", "2024-06-30")
}

# Helper function to pick a random date between start and end


def random_date(start, end):
    return pd.to_datetime(np.random.choice(pd.date_range(start, end)))


# Loop to generate each transaction
for i in range(num_transactions):
    # Randomly pick a customer
    customer = customers_df.sample(1).to_dict(as_series=False)

    # Randomly pick a product
    product = products_df.sample(1).to_dict(as_series=False)

    # 20% chance the purchase happens during a festival
    if random.random() < 0.2:
        fest = random.choice(list(festivals.keys()))
        purchase_date = random_date(festivals[fest][0], festivals[fest][1])
        festival = fest
    else:
        # Non-festival purchase
        purchase_date = random_date("2024-01-01", "2024-12-31")
        festival = None

    # Add the transaction record to the list
    transactions.append({
        "transaction_id": f"T{i}",                    # Unique transaction ID
        "customer_id": customer["customer_id"][0],   # Customer ID
        "product_id": product["product_id"][0],      # Product ID
        "purchase_date": purchase_date,              # Purchase date
        "price": product["price"][0],                # Product price
        "city": customer["city"][0],                 # Customer city
        "festival": festival                          # Festival name or None
    })

# Convert the list of transactions into a Polars DataFrame
transactions_df = pl.DataFrame(transactions)

# Save the transactions DataFrame to CSV inside the data folder
transactions_df.write_csv(data_folder / "transactions.csv")

# Display the first few rows of the transactions DataFrame
transactions_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival
str,str,str,datetime[μs],i64,str,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null


## Quick Exploration


In [8]:
# Display top products by number of transactions
print("Top Products by Sales")

print(
    transactions_df
    .group_by("product_id")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
    .head(5)
)


# Display cities with the most transactions
print("\nTop Cities by Transactions")

print(
    transactions_df
    .group_by("city")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
    .head(5)
)


# Display number of purchases during festivals
print("\nFestival Purchases")

print(
    transactions_df
    .group_by("festival")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
)

Top Products by Sales
shape: (5, 2)
┌────────────┬───────────────────┐
│ product_id ┆ transaction_count │
│ ---        ┆ ---               │
│ str        ┆ u32               │
╞════════════╪═══════════════════╡
│ p9         ┆ 5285              │
│ p8         ┆ 5234              │
│ p5         ┆ 5056              │
│ p0         ┆ 5028              │
│ p4         ┆ 5015              │
└────────────┴───────────────────┘

Top Cities by Transactions
shape: (5, 2)
┌──────────────┬───────────────────┐
│ city         ┆ transaction_count │
│ ---          ┆ ---               │
│ str          ┆ u32               │
╞══════════════╪═══════════════════╡
│ South Dumdum ┆ 446               │
│ Allahabad    ┆ 435               │
│ Berhampore   ┆ 421               │
│ Haldia       ┆ 411               │
│ Ghaziabad    ┆ 406               │
└──────────────┴───────────────────┘

Festival Purchases
shape: (4, 2)
┌──────────┬───────────────────┐
│ festival ┆ transaction_count │
│ ---      ┆ ---              

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")
C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:22: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")
C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:36: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")


In [9]:
# Verifying dataset size
print(transactions_df.shape)

(50000, 7)


# 02_day2_data_cleaning.ipynb
==================================================

## Imports


In [2]:
# Import Required Libraries
import polars as pl
from pathlib import Path

## Define Data Path


In [3]:
# Define the path to the data folder
data_folder = Path(
    "C:/Users/Mudit PC/OneDrive/Desktop/Data analytics projects/Indian_Beauty_Analytics/data")

## Load the datasets


In [4]:
# Load the customers dataset
customers_df = pl.read_csv(data_folder / "customers.csv")

# Load the products dataset
products_df = pl.read_csv(data_folder / "products.csv")

# Load the transactions dataset
transactions_df = pl.read_csv(data_folder / "transactions.csv")

## Inspecting the Datasets


In [5]:
# Display first rows of customers dataset
customers_df.head()

# Display first rows of products dataset
products_df.head()

# Display first rows of transactions dataset
transactions_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival
str,str,str,str,i64,str,str
"""T0""","""c1""","""p4""","""2024-09-23T00:00:00.000000""",399,"""Bhubaneswar""",null
"""T1""","""c948""","""p1""","""2024-11-12T00:00:00.000000""",499,"""Ghaziabad""",null
"""T2""","""c802""","""p6""","""2024-03-04T00:00:00.000000""",249,"""Serampore""",null
"""T3""","""c525""","""p9""","""2024-09-02T00:00:00.000000""",299,"""Ghaziabad""",null
"""T4""","""c488""","""p6""","""2024-08-06T00:00:00.000000""",249,"""Purnia""",null


## Check dataset structure


In [6]:
# Show schema of customers dataset
customers_df.schema

# Show schema of products dataset
products_df.schema

# Show schema of transactions dataset
transactions_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', String),
        ('price', Int64),
        ('city', String),
        ('festival', String)])

## Convert purchase date to datetime


In [7]:
# Convert purchase_date column to datetime format
transactions_df = transactions_df.with_columns(
    pl.col("purchase_date").str.to_datetime())

In [8]:
transactions_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', Datetime(time_unit='us', time_zone=None)),
        ('price', Int64),
        ('city', String),
        ('festival', String)])

## Check for null values


In [9]:
# check missing values in transactions dataset
transactions_df.null_count()

# check missing values in products dataset
products_df.null_count()

# check missing values in customers dataset
customers_df.null_count()

customer_id,name,city,age,gender
u32,u32,u32,u32,u32
0,0,0,0,0


## Validate dataset sizes


In [10]:
print(transactions_df.shape)

print(products_df.shape)

print(customers_df.shape)

(50000, 7)
(10, 4)
(1000, 5)


## Join product information to transactions

Transactions alone are not useful. we need product metadata


In [11]:
# Join transactions with products information
sales_df = transactions_df.join(
    products_df,
    on="product_id",
    how="left"
)

# Dropping the redundant columns
sales_df = sales_df.drop("price_right")

sales_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival,product_name,category
str,str,str,datetime[μs],i64,str,str,str,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null,"""Face Wash""","""Skincare"""
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null,"""Vitamin C Serum""","""Skincare"""
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null,"""Hair Oil""","""Haircare"""
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null,"""Face Pack""","""Skincare"""
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null,"""Hair Oil""","""Haircare"""


## Join customer information


In [12]:
# Join customer information
sales_df = sales_df.join(
    customers_df,
    on="customer_id",
    how="left"
)

# Dropping the redundant columns
sales_df = sales_df.drop("city_right")

sales_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival,product_name,category,name,age,gender
str,str,str,datetime[μs],i64,str,str,str,str,str,i64,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null,"""Face Wash""","""Skincare""","""Tanveer Kade""",43,"""Male"""
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null,"""Vitamin C Serum""","""Skincare""","""Christopher Rau""",50,"""Male"""
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null,"""Hair Oil""","""Haircare""","""Divya Krishnan""",30,"""Female"""
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null,"""Face Pack""","""Skincare""","""Jonathan Shroff""",54,"""Female"""
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null,"""Hair Oil""","""Haircare""","""Fariq Rai""",21,"""Male"""


## Create revenue column

Revenue is the core business metric


In [13]:
# Create revenue column based on product price
sales_df = sales_df.with_columns(
    pl.col("price").alias("revenue")
)

## Revenue by product


In [14]:
# Calculate revenue by products
sales_df.group_by("product_name").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

product_name,total_revenue
str,i64
"""Sunscreen SPF50""",3028544
"""Vitamin C Serum""",2466557
"""Face Wash""",2000985
"""Body Lotion""",1737671
"""Face Pack""",1580215
"""Aloe Vera Gel""",1503372
"""Hair Oil""",1210389
"""Nail Polish""",1041566
"""Herbal Shampoo""",980672


## Revenue by city


In [15]:
# Calculate revenue by city
sales_df.group_by("city").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

city,total_revenue
str,i64
"""South Dumdum""",144004
"""Berhampore""",138479
"""Allahabad""",137415
"""Ghaziabad""",133544
"""Haldia""",133489
…,…
"""Lucknow""",8619
"""Tumkur""",8224
"""Raiganj""",5634


## Revenue by festival


In [16]:
# Calculate revenue by fesitval

sales_df.group_by("festival").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

festival,total_revenue
str,i64
null,13009013
"""Holi""",1097119
"""Diwali""",1088336
"""Summer""",1051482


## Check top selling products


In [17]:
# Count number of transcations per product

sales_df.group_by("product_name").count().sort("count", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_14736\3571595247.py:3: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  sales_df.group_by("product_name").count().sort("count", descending=True)


product_name,count
str,u32
"""Face Pack""",5285
"""Nail Polish""",5234
"""Sunscreen SPF50""",5056
"""Aloe Vera Gel""",5028
"""Face Wash""",5015
"""Body Lotion""",4979
"""Vitamin C Serum""",4943
"""Herbal Shampoo""",4928
"""Hair Oil""",4861


## Reordering for better readability


In [18]:
# Reorder columns for cleaner analytics structure
sales_df = sales_df.select([
    "transaction_id",
    "customer_id",
    "product_id",
    "purchase_date",
    "festival",
    "city",
    "product_name",
    "category",
    "name",
    "age",
    "gender",
    "price",
    "revenue"
])

In [19]:
# Renaming name column for better readability
sales_df = sales_df.rename({
    "name": "customer_name"
})

In [20]:
sales_df.select(pl.col("transaction_id").n_unique())

transaction_id
u32
50000


In [21]:
sales_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', Datetime(time_unit='us', time_zone=None)),
        ('festival', String),
        ('city', String),
        ('product_name', String),
        ('category', String),
        ('customer_name', String),
        ('age', Int64),
        ('gender', String),
        ('price', Int64),
        ('revenue', Int64)])

In [22]:
sales_df = sales_df.sort("purchase_date")

## Saving the data set in csv file


In [23]:
# Save cleaned sales dataset
sales_df.write_csv(data_folder / "sales_cleaned.csv")

## Saving the data set in parquet


In [24]:
sales_df.write_parquet(data_folder / "sales_cleaned.parquet")

# 03_day3_rfm_analysis.ipynb
==================================================

## Importing libraries


In [2]:
# Import Polars for dataframe operations
import polars as pl

# Import Path for managing file paths
from pathlib import Path

## Define Data folder


In [3]:
# Define path to data folder
data_folder = Path("../data")

## Load Cleaned Dataset (Parquet Version)


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Check data types
print(sales_df.schema)

# preview data set
sales_df.head()

print("Rows:", sales_df.height)
print("Columns:", sales_df.width)

Schema({'transaction_id': String, 'customer_id': String, 'product_id': String, 'purchase_date': Datetime(time_unit='us', time_zone=None), 'festival': String, 'city': String, 'product_name': String, 'category': String, 'customer_name': String, 'age': Int64, 'gender': String, 'price': Int64, 'revenue': Int64})
Rows: 50000
Columns: 13


## Find Latest Purchase Date


In [10]:
# Find the most recent purchase date
latest_date = sales_df.select(pl.col("purchase_date").max()).item()
latest_date

datetime.datetime(2024, 12, 31, 0, 0)

## Calculate Recency

Recency = Days since last purchase


In [12]:
# Get last purchase date per customer
receny_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_date").max().alias("last_purchase")
)

# Calculate recency

receny_df = receny_df.with_columns(
    (latest_date - pl.col("last_purchase")).dt.total_days().alias("recency")
)

receny_df.head()

customer_id,last_purchase,recency
str,datetime[μs],i64
"""c667""",2024-12-31 00:00:00,0
"""c521""",2024-12-26 00:00:00,5
"""c204""",2024-12-13 00:00:00,18
"""c96""",2024-12-17 00:00:00,14
"""c420""",2024-12-24 00:00:00,7


## Calculate Frequency

Frequency = number of purchases


In [13]:
# Count purchases per customer

frequency_df = sales_df.group_by("customer_id").agg(
    pl.count().alias("frequency")
)

frequency_df.head()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\2495761181.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("frequency")


customer_id,frequency
str,u32
"""c538""",79
"""c349""",45
"""c459""",58
"""c688""",61
"""c775""",55


## Calculate Monetary Value

Monetary = total revenue spent


In [17]:
# Calculate total revenue per customer

monetary_df = sales_df.group_by("customer_id").agg(
    pl.col("revenue").sum().alias("monetary")
)

monetary_df.head()

customer_id,monetary
str,i64
"""c517""",16845
"""c840""",24627
"""c994""",18437
"""c607""",20886
"""c105""",20239


## Merge RFM Metrics


In [18]:
# Merge recency, frequency and monetary tables
rfm_df = receny_df.join(frequency_df, on="customer_id").join(
    monetary_df, on="customer_id"
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary
str,datetime[μs],i64,u32,i64
"""c517""",2024-12-28 00:00:00,3,55,16845
"""c840""",2024-12-29 00:00:00,2,73,24627
"""c994""",2024-12-30 00:00:00,1,63,18437
"""c607""",2024-12-27 00:00:00,4,64,20886
"""c105""",2024-12-15 00:00:00,16,61,20239


## Create RFM Scores


In [20]:
# Create RFM scores using rank

rfm_df = rfm_df.with_columns(
    [
        pl.col("recency").rank("dense").alias("r_score"),
        pl.col("frequency").rank("dense").alias("f_score"),
        pl.col("monetary").rank("dense").alias("m_score"),
    ]
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score
str,datetime[μs],i64,u32,i64,u32,u32,u32
"""c517""",2024-12-28 00:00:00,3,55,16845,4,47,492
"""c840""",2024-12-29 00:00:00,2,73,24627,3,65,806
"""c994""",2024-12-30 00:00:00,1,63,18437,2,55,581
"""c607""",2024-12-27 00:00:00,4,64,20886,5,56,694
"""c105""",2024-12-15 00:00:00,16,61,20239,17,53,675


## Create RFM Score


In [ ]:
# Combine RFM scores

rfm_df = rfm_df.with_columns(
    (
        pl.col("r_score") +
        pl.col("f_score") +
        pl.col("m_score")
    )
    .alias("rfm_score")
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
str,datetime[μs],i64,u32,i64,u32,u32,u32,u32
"""c517""",2024-12-28 00:00:00,3,55,16845,543,47,492,1082
"""c840""",2024-12-29 00:00:00,2,73,24627,874,65,806,1745
"""c994""",2024-12-30 00:00:00,1,63,18437,638,55,581,1274
"""c607""",2024-12-27 00:00:00,4,64,20886,755,56,694,1505
"""c105""",2024-12-15 00:00:00,16,61,20239,745,53,675,1473


## Create Customer Segments


In [ ]:
# Assign customer segments

rfm_df = rfm_df.with_columns(
    pl.when(pl.col("rfm_score") >= 1500).then(pl.lit("Champions"))
    .when(pl.col("rfm_score") >= 1200).then(pl.lit("Loyal Customers"))
    .when(pl.col("rfm_score") >= 600).then(pl.lit("Potential Loyalists"))
    .otherwise(pl.lit("At Risk"))
    .alias("segment")
)

rfm_df.head()

customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
str,datetime[μs],i64,u32,i64,u32,u32,u32,u32,str
"""c517""",2024-12-28 00:00:00,3,55,16845,543,47,492,1082,"""Potential Loyalists"""
"""c840""",2024-12-29 00:00:00,2,73,24627,874,65,806,1745,"""Champions"""
"""c994""",2024-12-30 00:00:00,1,63,18437,638,55,581,1274,"""Loyal Customers"""
"""c607""",2024-12-27 00:00:00,4,64,20886,755,56,694,1505,"""Champions"""
"""c105""",2024-12-15 00:00:00,16,61,20239,745,53,675,1473,"""Loyal Customers"""


## Segmet Distribution


In [34]:
# Count customers per segment

rfm_df.group_by("segment").count().sort("count", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\3515162173.py:3: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rfm_df.group_by("segment").count().sort("count", descending=True)


segment,count
str,u32
"""Potential Loyalists""",327
"""At Risk""",302
"""Champions""",206
"""Loyal Customers""",165


## Save Dataset


In [ ]:
# Save RFM dataframe as csv
rfm_df.write_csv(data_folder / "customer_rfm.csv")

In [36]:
# Save RFM dataframe as parquet
rfm_df.write_parquet(data_folder / "customer_rfm.parquet")

# Sanity Check

In [37]:
rfm_df.describe()

statistic,customer_id,last_purchase,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
str,str,str,f64,f64,f64,f64,f64,f64,f64,str
"""count""","""1000""","""1000""",1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,"""1000"""
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,"""2024-12-21 17:00:57.600000""",9.291,50.0,16245.95,490.591,41.96,438.415,970.966,null
"""std""",null,null,10.69461,17.516702,5803.458224,267.482902,17.382363,253.909698,538.144211,null
"""min""","""c0""","""2024-10-23 00:00:00""",0.0,7.0,2493.0,6.0,1.0,1.0,8.0,"""At Risk"""
"""25%""",null,"""2024-12-18 00:00:00""",2.0,38.0,12214.0,263.0,30.0,219.0,513.0,null
"""50%""",null,"""2024-12-26 00:00:00""",6.0,49.0,15955.0,490.0,41.0,439.0,973.0,null
"""75%""",null,"""2024-12-29 00:00:00""",13.0,61.0,19691.0,712.0,53.0,653.0,1416.0,null
"""max""","""c999""","""2024-12-31 00:00:00""",69.0,110.0,36590.0,988.0,93.0,891.0,1972.0,"""Potential Loyalists"""


In [38]:
rfm_df.group_by("segment").count()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_3456\84912235.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rfm_df.group_by("segment").count()


segment,count
str,u32
"""Loyal Customers""",165
"""Champions""",206
"""At Risk""",302
"""Potential Loyalists""",327


# 04_day4_cohort_retention.ipynb
==================================================

# Importing Libraries


In [1]:
# Import Polars for dataframe operations
import polars as pl

# Import Path to manage fiel paths
from pathlib import Path

## Define Data Path


In [2]:
# Define path to the data folder
data_folder = Path("../data")

# Load Cleaned Dataset


In [3]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Extract Purchase Month

skipped purchase_date to datetime since it already exists.


In [6]:
# Extract purchase month
sales_df = sales_df.with_columns(
    pl.col("purchase_date")
    .dt.truncate("1mo")
    .alias("purchase_month")
)

# Quick Check
sales_df.select(["purchase_date", "purchase_month"]).head()

purchase_date,purchase_month
datetime[μs],datetime[μs]
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00
2024-01-01 00:00:00,2024-01-01 00:00:00


# Identify Customer First Purchase (Cohort)

Each customer belongs to the month they first purchased.


In [9]:
# Find first purchase month for each customer
cohort_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_month").min().alias("cohort_month")
)

cohort_df.head()

customer_id,cohort_month
str,datetime[μs]
"""c198""",2024-01-01 00:00:00
"""c183""",2024-01-01 00:00:00
"""c160""",2024-01-01 00:00:00
"""c749""",2024-01-01 00:00:00
"""c167""",2024-01-01 00:00:00


## Join Cohort Back to Transactions


In [ ]:
# Merge cohort info into sales data
sales_cohort_df = sales_df.join(cohort_df, on="customer_id")

# Quick Check
sales_cohort_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,purchase_month,cohort_month
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,datetime[μs],datetime[μs]
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,2024-01-01 00:00:00,2024-01-01 00:00:00
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00


## Calculating Cohort Index

cohort index = months since first purchase


In [ ]:
# Calculate months since first purchase
sales_cohort_df = sales_cohort_df.with_columns(
    (
        (pl.col("purchase_month").dt.year() -
         pl.col("cohort_month").dt.year()) * 12
        + (pl.col("purchase_month").dt.month() -
           pl.col("cohort_month").dt.month())
    ).alias("cohort_index")
)

sales_cohort_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,purchase_month,cohort_month,cohort_index
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,datetime[μs],datetime[μs],i32
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,2024-01-01 00:00:00,2024-01-01 00:00:00,0
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,2024-01-01 00:00:00,2024-01-01 00:00:00,0


In [ ]:
# Quick Check
sales_cohort_df.select(
    ["purchase_month", "cohort_month", "cohort_index"]).tail(10)

purchase_month,cohort_month,cohort_index
datetime[μs],datetime[μs],i32
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11
2024-12-01 00:00:00,2024-01-01 00:00:00,11


## Count Active Customers Per Cohort


In [16]:
# Count customers in each cohort period
cohort_counts = sales_cohort_df.group_by(
    ["cohort_month", "cohort_index"]
).agg(pl.col("customer_id").n_unique().alias("customers"))

cohort_counts.tail()

cohort_month,cohort_index,customers
datetime[μs],i32,u32
2024-03-01 00:00:00,8,7
2024-03-01 00:00:00,2,8
2024-01-01 00:00:00,4,911
2024-01-01 00:00:00,9,923
2024-01-01 00:00:00,8,873


## Get Cohort Sizes


In [ ]:
# Get initial cohort size
cohort_sizes = cohort_counts.filter(pl.col("cohort_index") == 0
                                    ).select(["cohort_month", "customers"]
                                             ).rename({"customers": "cohort_size"})

## Calculate Retention Rate


In [21]:
# Merge cohort size
cohort_retention = cohort_counts.join(
    cohort_sizes, on="cohort_month"
)

# Calculate retention rate
cohort_retention = cohort_retention.with_columns(
    (pl.col("customers") / pl.col("cohort_size")).alias("retention_rate")
)

cohort_retention.head()

cohort_month,cohort_index,customers,cohort_size,retention_rate
datetime[μs],i32,u32,u32,f64
2024-01-01 00:00:00,5,917,934,0.981799
2024-02-01 00:00:00,0,57,57,1.0
2024-06-01 00:00:00,4,1,1,1.0
2024-02-01 00:00:00,10,52,57,0.912281
2024-01-01 00:00:00,0,934,934,1.0


## Save Dataset


In [ ]:
# Save cohort retention dataset
cohort_retention.write_csv(data_folder / "cohort_retention.csv")

# Save parquet version
cohort_retention.write_parquet(data_folder / "cohort_retention.parquet")

# 05_day5_customer_behavior.ipynb
==================================================

## Import Libraries


In [2]:
# Import Polars for dataframe analytics
import polars as pl
# Improt Path to manage file paths
from pathlib import Path

## Define Data Path


In [3]:
# Define data folder path
data_folder = Path("../data")

## Load Cleaned Dataset


In [4]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Create Customer-Level Metrics


In [5]:
# Aggregate customer level behavior
customer_metrics = (sales_df.group_by("customer_id").agg(

    # Total orders per customer
    pl.len().alias("total_orders"),

    # Total money spent
    pl.col("revenue").sum().alias("total_spent"),

    # Average order value
    pl.col("revenue").mean().alias("avg_order_value"),

    # First Purchase date
    pl.col("purchase_date").min().alias("first_purchase"),

    # Last purchase date
    pl.col("purchase_date").max().alias("last_purchase")

)
)
customer_metrics.head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
str,u32,i64,f64,datetime[μs],datetime[μs]
"""c970""",50,16850,337.0,2024-01-01 00:00:00,2024-12-10 00:00:00
"""c935""",73,23277,318.863014,2024-01-05 00:00:00,2024-12-31 00:00:00
"""c486""",48,15652,326.083333,2024-01-11 00:00:00,2024-11-28 00:00:00
"""c589""",69,21231,307.695652,2024-01-11 00:00:00,2024-12-30 00:00:00
"""c717""",48,15052,313.583333,2024-01-03 00:00:00,2024-12-20 00:00:00


## Calculate Customer Lifetime


In [6]:
# Calculate customer lifetime in days
customer_metrics = customer_metrics.with_columns(
    (pl.col("last_purchase") - pl.col("first_purchase")
     ).dt.total_days().alias("customer_lifetime_days")
)

customer_metrics.tail(10)

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days
str,u32,i64,f64,datetime[μs],datetime[μs],i64
"""c249""",47,15153,322.404255,2024-01-21 00:00:00,2024-12-10 00:00:00,324
"""c810""",48,16502,343.791667,2024-01-05 00:00:00,2024-12-25 00:00:00,355
"""c308""",11,3389,308.090909,2024-01-27 00:00:00,2024-10-27 00:00:00,274
"""c338""",55,17595,319.909091,2024-01-01 00:00:00,2024-12-27 00:00:00,361
"""c797""",43,13707,318.767442,2024-01-06 00:00:00,2024-12-31 00:00:00,360
"""c475""",59,21291,360.864407,2024-01-08 00:00:00,2024-12-31 00:00:00,358
"""c374""",96,33154,345.354167,2024-01-06 00:00:00,2024-12-27 00:00:00,356
"""c704""",38,14012,368.736842,2024-01-04 00:00:00,2024-12-29 00:00:00,360
"""c741""",42,13858,329.952381,2024-01-20 00:00:00,2024-12-31 00:00:00,346


## Calculate Purchase Frequency


In [7]:
# Purchase frequecy = orders / lifetime
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days")+1
                                  )
    ).alias("purchase_frequency")
)

# Note:- adding +1 avoids divide by 0 for single purchase customer
customer_metrics.head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,u32,i64,f64,datetime[μs],datetime[μs],i64,f64
"""c970""",50,16850,337.0,2024-01-01 00:00:00,2024-12-10 00:00:00,344,0.144928
"""c935""",73,23277,318.863014,2024-01-05 00:00:00,2024-12-31 00:00:00,361,0.201657
"""c486""",48,15652,326.083333,2024-01-11 00:00:00,2024-11-28 00:00:00,322,0.148607
"""c589""",69,21231,307.695652,2024-01-11 00:00:00,2024-12-30 00:00:00,354,0.194366
"""c717""",48,15052,313.583333,2024-01-03 00:00:00,2024-12-20 00:00:00,352,0.135977


# Save Customer Level Dataset

This dataset will later power segmentation and churn analysis


In [8]:
# Save customer level metrics
customer_metrics.write_csv(data_folder / "customer_level_metrics.csv")

# Save customer level metrics in parquet
customer_metrics.write_parquet(data_folder / "customer_level_metrics.parquet")

## Calculate Global KPIs

Computing business-level metrics.


In [9]:
# Total customers
total_customers = sales_df.select(
    pl.col("customer_id").n_unique()
).item()

# Total orders
total_orders = sales_df.height

# Total revenue
total_revenue = sales_df.select(
    pl.col("revenue").sum()
).item()

## Calculate Ecommerce Metrics


In [10]:
# Average order value
aov = total_revenue / total_orders

# Purhcase frequency
purchase_frequency = total_orders / total_customers

# Revenue per customer
revenue_per_customer = total_revenue / total_customers

## Repeat Purchase Rate


In [11]:
# Customers with multiple purchases
repeat_customers = customer_metrics.filter(
    pl.col("total_orders") > 1
).height

repeat_purchase_rate = repeat_customers / total_customers

print("Repeat Customer:", repeat_customers)
print("Repeat Purchase Rate:", repeat_purchase_rate)

Repeat Customer: 1000
Repeat Purchase Rate: 1.0


## Create KPI Tables


In [12]:
# Create KPI dataframe
metrics_df = pl.DataFrame(
    {
        "metric": [
            "Total Customers",
            "Total Orders",
            "Total Revenue",
            "Average Order Value",
            "Purchase Frequency",
            "Revenue per Customer",
            "Repeat Customers",
            "Repeat Purchase Rate"
        ],
        "value": [
            float(total_customers),
            float(total_orders),
            float(total_revenue),
            round(aov, 2),
            round(purchase_frequency, 2),
            round(revenue_per_customer, 2),
            round(repeat_customers, 2),
            round(repeat_purchase_rate, 2)
        ]
    }
)

metrics_df

metric,value
str,f64
"""Total Customers""",1000.0
"""Total Orders""",50000.0
"""Total Revenue""",1.624595e7
"""Average Order Value""",324.92
"""Purchase Frequency""",50.0
"""Revenue per Customer""",16245.95
"""Repeat Customers""",1000.0
"""Repeat Purchase Rate""",1.0


In [13]:
metrics_df.sort("value", descending=True)

metric,value
str,f64
"""Total Revenue""",1.624595e7
"""Total Orders""",50000.0
"""Revenue per Customer""",16245.95
"""Total Customers""",1000.0
"""Repeat Customers""",1000.0
"""Average Order Value""",324.92
"""Purchase Frequency""",50.0
"""Repeat Purchase Rate""",1.0


In [14]:
customer_metrics.sort("total_orders", descending=True).head()

customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,u32,i64,f64,datetime[μs],datetime[μs],i64,f64
"""c102""",110,36590,332.636364,2024-01-14 00:00:00,2024-12-28 00:00:00,349,0.314286
"""c22""",109,35191,322.853211,2024-01-04 00:00:00,2024-12-26 00:00:00,357,0.304469
"""c821""",104,34296,329.769231,2024-01-04 00:00:00,2024-12-30 00:00:00,361,0.287293
"""c860""",103,34947,339.291262,2024-01-07 00:00:00,2024-12-29 00:00:00,357,0.287709
"""c937""",101,32849,325.237624,2024-01-01 00:00:00,2024-12-31 00:00:00,365,0.275956


In [15]:
customer_metrics.describe()

statistic,customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency
str,str,f64,f64,f64,str,str,f64,f64
"""count""","""1000""",1000.0,1000.0,1000.0,"""1000""","""1000""",1000.0,1000.0
"""null_count""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,0.0
"""mean""",null,50.0,16245.95,324.785374,"""2024-01-11 07:32:09.600000""","""2024-12-21 17:00:57.600000""",345.395,0.143393
"""std""",null,17.516702,5803.458224,20.911029,null,null,17.474162,0.047757
"""min""","""c0""",7.0,2493.0,260.538462,"""2024-01-01 00:00:00""","""2024-10-23 00:00:00""",205.0,0.033981
"""25%""",null,38.0,12214.0,310.111111,"""2024-01-03 00:00:00""","""2024-12-18 00:00:00""",339.0,0.109145
"""50%""",null,49.0,15955.0,324.892857,"""2024-01-07 00:00:00""","""2024-12-26 00:00:00""",350.0,0.142012
"""75%""",null,61.0,19691.0,339.0,"""2024-01-15 00:00:00""","""2024-12-29 00:00:00""",358.0,0.173021
"""max""","""c999""",110.0,36590.0,394.833333,"""2024-06-06 00:00:00""","""2024-12-31 00:00:00""",365.0,0.314286


In [16]:
customer_metrics.select([
    pl.col("total_orders").min().alias("min_orders"),
    pl.col("total_orders").mean().alias("avg_orders"),
    pl.col("total_orders").max().alias("max_orders")
])

min_orders,avg_orders,max_orders
u32,f64,u32
7,50.0,110


## Save Metrics Dataset


In [17]:
# Save KPI Dataset
metrics_df.write_csv(data_folder / "customer_behavior_metrics.csv")

metrics_df.write_parquet(data_folder / "customer_behavior_metrics.parquet")

# Adding Customer Segmentation

In [36]:
customer_metrics = customer_metrics.with_columns(

    pl.when(pl.col("total_spent") > 30000).then(pl.lit("VIP"))
    .when(pl.col("total_orders") >= 30).then(pl.lit("Loyal"))
    .when(pl.col("total_orders") >= 10).then(pl.lit("Regular"))
    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")

)

In [37]:
customer_metrics.group_by("customer_type").count()

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_20912\2212270806.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  customer_metrics.group_by("customer_type").count()


customer_type,count
str,u32
"""Regular""",111
"""Low Value""",2
"""Loyal""",874
"""VIP""",13


In [38]:
sales_encriched = sales_df.join(
    customer_metrics.select(["customer_id", "customer_type"]),
    on="customer_id",
    how="left"
)

In [39]:
sales_encriched.select("customer_type").unique()

customer_type
str
"""Loyal"""
"""Low Value"""
"""VIP"""
"""Regular"""


In [40]:
sales_encriched.write_parquet(data_folder / "sales_encriched.parquet")

# 06_day6_product_performance.ipynb
==================================================

# Importing Libraries

In [100]:
# Import polars for analytics
import polars as pl

# Path for data folder
from pathlib import Path

# Define Data Path

In [101]:
# Define data folder
data_folder = Path("../data")

# Loading the Dataset

In [102]:
# Load cleaned dataset
sales_df = pl.read_parquet(data_folder /"sales_cleaned.parquet")

# Preview dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


# Product Performance

In [103]:
# Calculate product performance
product_performance = (
    sales_df.group_by(["product_id","product_name","category"])
    .agg(
        # number of transactions
        pl.len().alias("transactions"),

        # total revenue
        pl.col("revenue").sum().alias("total_revenue"),

        # average product price
        pl.col("revenue").mean().alias("avg_price")

    )
)

product_performance.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0


# Sort Top and Bottom Products

In [104]:
# Sort top products by revenue
top_products = (product_performance.sort("total_revenue",descending=True))

top_products.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0


In [105]:
# Sort Bottom products by revenue
bottom_products = (product_performance.sort("total_revenue",descending=False))

bottom_products.head()

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p3""","""Lip Balm""","""Makeup""",4671,695979,149.0
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0


# Category Performance

In [106]:
# Revenue by category
category_performance = (
    sales_df.group_by(["category"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

category_performance.sort("total_revenue",descending = True).head()

category,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Skincare""",30306,12317344,406.432522
"""Haircare""",9789,2191061,223.82889
"""Makeup""",9905,1737545,175.420999


# City Performance

In [107]:
# Revenue by city
city_performance = (
    sales_df.group_by(["city"])
    .agg(
        pl.len().alias("transactions"),
        
        pl.col("revenue").sum().alias("total_revenue"),

        pl.col("revenue").mean().alias("avg_order_value")
    )
    .sort("total_revenue", descending=True)
)

city_performance.sort("avg_order_value",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Bongaigaon""",38,14012,368.736842
"""Bhavnagar""",49,17751,362.265306
"""Motihari""",77,27873,361.987013
"""New Delhi""",83,29767,358.638554
"""Srikakulam""",78,27722,355.410256


In [108]:
city_performance.sort("total_revenue",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,322.878924
"""Berhampore""",421,138479,328.928741
"""Allahabad""",435,137415,315.896552
"""Ghaziabad""",406,133544,328.926108
"""Haldia""",411,133489,324.790754


In [109]:
city_performance.sort("transactions",descending = True).head()

city,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,322.878924
"""Allahabad""",435,137415,315.896552
"""Berhampore""",421,138479,328.928741
"""Haldia""",411,133489,324.790754
"""Ghaziabad""",406,133544,328.926108


## Pareto Analysis (80/20 Rule)
Checking wether few products drive most revenue

In [110]:
# Add revenue share 
products_pareto = (
    top_products.with_columns(
        (pl.col("total_revenue") / 
        pl.col("total_revenue").sum()
        ).alias("revenue_share")
    )
)

products_pareto.sort("revenue_share",descending = True)

product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share
str,str,str,u32,i64,f64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,0.186418
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,0.151826
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,0.123168
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,0.10696
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,0.097268
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,0.092538
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,0.074504
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,0.064112
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,0.060364


## Revenue Share

In [111]:
product_performance = product_performance.with_columns(
    (pl.col("total_revenue") / pl.col("total_revenue").sum()*100).round(2).alias("revenue_share_pct" )
)

product_performance.sort("revenue_share_pct",descending = True)

product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share_pct
str,str,str,u32,i64,f64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,18.64
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,15.18
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,12.32
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,10.7
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,9.73
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,9.25
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,7.45
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,6.41
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,6.04


# Adding cumulative column to the product table

In [112]:
# Sort products by revenue
product_performance = product_performance.sort(
    "total_revenue",descending=True    
)

# Add cumulative revenue column
product_performance = product_performance.with_columns(

    # cumulative revenue
    pl.col("total_revenue").cum_sum().alias("cumulative_revenue"),

    # cumulative revenue percentage
    (
        pl.col("total_revenue").cum_sum()
        /
        pl.col("total_revenue").sum()
        * 100
    ).round(2).alias("cumulative_revenue_pct")
)

# view table
product_performance


product_id,product_name,category,transactions,total_revenue,avg_price,revenue_share_pct,cumulative_revenue,cumulative_revenue_pct
str,str,str,u32,i64,f64,f64,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0,18.64,3028544,18.64
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0,15.18,5495101,33.82
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0,12.32,7496086,46.14
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0,10.7,9233757,56.84
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0,9.73,10813972,66.56
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0,9.25,12317344,75.82
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0,7.45,13527733,83.27
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0,6.41,14569299,89.68
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0,6.04,15549971,95.72


## Save the Datasets

In [113]:
# Save product dataset
product_performance.write_parquet(data_folder/"product_performance.parquet")

# Save category dataset
category_performance.write_parquet(data_folder/"category_performance.parquet")

# Save city dataset
city_performance.write_parquet(data_folder/"city_performance.parquet")

# Save top products dataset
top_products.write_parquet(data_folder/"top_products.parquet")

In [114]:
top_products.head(10)

product_id,product_name,category,transactions,total_revenue,avg_price
str,str,str,u32,i64,f64
"""p5""","""Sunscreen SPF50""","""Skincare""",5056,3028544,599.0
"""p1""","""Vitamin C Serum""","""Skincare""",4943,2466557,499.0
"""p4""","""Face Wash""","""Skincare""",5015,2000985,399.0
"""p7""","""Body Lotion""","""Skincare""",4979,1737671,349.0
"""p9""","""Face Pack""","""Skincare""",5285,1580215,299.0
"""p0""","""Aloe Vera Gel""","""Skincare""",5028,1503372,299.0
"""p6""","""Hair Oil""","""Haircare""",4861,1210389,249.0
"""p8""","""Nail Polish""","""Makeup""",5234,1041566,199.0
"""p2""","""Herbal Shampoo""","""Haircare""",4928,980672,199.0


In [115]:
category_performance

category,transactions,total_revenue,avg_order_value
str,u32,i64,f64
"""Skincare""",30306,12317344,406.432522
"""Haircare""",9789,2191061,223.82889
"""Makeup""",9905,1737545,175.420999


In [116]:
product_performance.select(pl.col("total_revenue").sum())

total_revenue
i64
16245950


# 07_day7_festival_analysis.ipynb
==================================================

# Imports


In [32]:
# Import required libraries
import polars as pl
from pathlib import Path

# Data Path


In [33]:
# Define data folder path
data_folder = Path("../data")

# Load Dataset


In [34]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview data
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Festival vs Non-Festival Sales


In [35]:
# Revenue comparison between festival and normal days
festival_sales = sales_df.group_by("festival").agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue"),

    pl.col("revenue").mean().round().alias("avg_order_value")
)

festival_sales.sort("total_revenue", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\425732562.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,transactions,total_revenue,avg_order_value
str,u32,i64,f64
null,40087,13009013,325.0
"""Holi""",3331,1097119,329.0
"""Diwali""",3364,1088336,324.0
"""Summer""",3218,1051482,327.0


## Festival Revenue Share


In [36]:
festival_sales = festival_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round(1).alias("revenue_share_pct")
)

festival_sales

festival,transactions,total_revenue,avg_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""Summer""",3218,1051482,327.0,6.5
"""Holi""",3331,1097119,329.0,6.8
"""Diwali""",3364,1088336,324.0,6.7
null,40087,13009013,325.0,80.1


## Festival Category Performance


In [37]:
festival_category = sales_df.group_by(
    ["festival", "category"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_category

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\1494158052.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,category,transactions,total_revenue
str,str,u32,i64
null,"""Skincare""",24232,9848418
null,"""Haircare""",7849,1757101
null,"""Makeup""",8006,1403494
"""Diwali""","""Skincare""",2061,830189
"""Diwali""","""Haircare""",631,140369
…,…,…,…
"""Holi""","""Haircare""",676,152374
"""Holi""","""Makeup""",617,109233
"""Summer""","""Skincare""",1975,803225


# City Festival Performance


In [38]:
festival_city = sales_df.group_by(
    ["festival", "city"]
).agg(
    pl.count().alias("transactions"),

    pl.col("revenue").sum().alias("total_revenue")
).sort(
    ["festival", "total_revenue"],
    descending=[False, True]
)

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22628\820523211.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


In [39]:
import polars as pl

# Aggregate sales by city
city_summary = (
    sales_df.group_by("city")
    .agg([
        pl.count("transaction_id").alias(
            "transactions"),  # total transactions per city
        # total revenue per city
        pl.sum("revenue").alias("total_revenue")
    ])
    .with_columns(
        # average revenue per transaction
        (pl.col("total_revenue") / pl.col("transactions")
         ).round().alias("avg_revenue_per_txn")
    )
)

city_summary.sort("avg_revenue_per_txn", descending=True)

city,transactions,total_revenue,avg_revenue_per_txn
str,u32,i64,f64
"""Bongaigaon""",38,14012,369.0
"""Motihari""",77,27873,362.0
"""Bhavnagar""",49,17751,362.0
"""New Delhi""",83,29767,359.0
"""Ajmer""",172,60978,355.0
…,…,…,…
"""Sangli-Miraj & Kupwad""",105,30695,292.0
"""Ludhiana""",50,14350,287.0
"""Khammam""",43,12307,286.0


## Save the Data set


In [40]:
festival_sales.write_parquet(data_folder/"festival_sales.parquet")

In [41]:
festival_sales

festival,transactions,total_revenue,avg_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""Summer""",3218,1051482,327.0,6.5
"""Holi""",3331,1097119,329.0,6.8
"""Diwali""",3364,1088336,324.0,6.7
null,40087,13009013,325.0,80.1


# 08_day8_city_analysis.ipynb
==================================================

## Imports

In [12]:
import polars as pl
from pathlib import Path

# Data Path

In [13]:
data_folder = Path("../data")

## Load Dataset

In [14]:
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Total Revenue and Transactions by City

In [15]:
city_performance = sales_df.group_by(["city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("total_revenue", descending=True)

city_performance

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\253708900.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


city,transactions,total_revenue,average_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,323.0
"""Berhampore""",421,138479,329.0
"""Allahabad""",435,137415,316.0
"""Ghaziabad""",406,133544,329.0
"""Haldia""",411,133489,325.0
…,…,…,…
"""Lucknow""",31,8619,278.0
"""Tumkur""",26,8224,316.0
"""Raiganj""",16,5634,352.0


## Festival Sales by City

In [16]:
festival_city = sales_df.group_by(["festival", "city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\361083668.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


# City Revenue Share

In [19]:
city_performance = city_performance.with_columns(
   (
     pl.col("total_revenue")/pl.col("total_revenue").sum() * 100 
   ).round(1).alias("revenue_share_pct")
)

city_performance.sort("revenue_share_pct",descending = True)

city,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""South Dumdum""",446,144004,323.0,0.9
"""Berhampore""",421,138479,329.0,0.9
"""Allahabad""",435,137415,316.0,0.8
"""Ghaziabad""",406,133544,329.0,0.8
"""Haldia""",411,133489,325.0,0.8
…,…,…,…,…
"""Lucknow""",31,8619,278.0,0.1
"""Tumkur""",26,8224,316.0,0.1
"""Raiganj""",16,5634,352.0,0.0


In [20]:
festival_city = sales_df.group_by(["festival", "city"]).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\1709528429.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


# Save Outputs

In [21]:
city_performance.write_parquet(data_folder / "city_performance.parquet")
festival_city.write_parquet(data_folder / "festival_city.parquet")

# 09_day9_sales_trends.ipynb
==================================================

## Imports


In [18]:
import polars as pl
from pathlib import Path

# Data Path


In [19]:
data_folder = Path("../data")

# Load Dataset


In [20]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Create Year-Month Column


In [21]:
# Extract year-month for aggregation
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%Y-%m").alias("year_month")
)

# Monthly Sales Trends


In [22]:
monthly_sales = sales_df.group_by("year_month").agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("year_month")

monthly_sales

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22324\2184171659.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


year_month,transactions,total_revenue,average_order_value
str,u32,i64,f64
"""2024-01""",3297,1069903,325.0
"""2024-02""",3230,1051770,326.0
"""2024-03""",6672,2191528,328.0
"""2024-04""",3238,1042062,322.0
"""2024-05""",5068,1643682,324.0
…,…,…,…
"""2024-08""",3421,1115879,326.0
"""2024-09""",3262,1064738,326.0
"""2024-10""",5709,1844991,323.0


# Revenue Share by Month


In [23]:
monthly_sales = monthly_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round().alias("revenue_share_pct")
)

monthly_sales.sort("revenue_share_pct", descending=True)

year_month,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""2024-03""",6672,2191528,328.0,13.0
"""2024-10""",5709,1844991,323.0,11.0
"""2024-05""",5068,1643682,324.0,10.0
"""2024-06""",4970,1617130,325.0,10.0
"""2024-11""",4315,1394035,323.0,9.0
…,…,…,…,…
"""2024-08""",3421,1115879,326.0,7.0
"""2024-09""",3262,1064738,326.0,7.0
"""2024-12""",3439,1111261,323.0,7.0


# Festival Monthly Trends


In [ ]:
monthly_festival_sales = sales_df.group_by(["year_month", "festival"]
                                           ).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["year_month", "total_revenue"], descending=[False, True])

monthly_festival_sales.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22324\2365063454.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


year_month,festival,transactions,total_revenue
str,str,u32,i64
"""2024-01""",null,3297,1069903
"""2024-02""",null,3230,1051770
"""2024-03""","""Holi""",3331,1097119
"""2024-03""",null,3341,1094409
"""2024-04""",null,3238,1042062
"""2024-05""",null,3451,1113199
"""2024-05""","""Summer""",1617,530483
"""2024-06""",null,3369,1096131
"""2024-06""","""Summer""",1601,520999


# Adding month column to sales_df


In [25]:
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%B").alias("month_name")
)

# Save Outputs


In [ ]:
monthly_festival_sales.write_parquet(
    data_folder / "monthly_festival_sales.parquet")
monthly_sales.write_parquet(data_folder / "monthly_sales.parquet")

In [27]:
monthly_sales

year_month,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""2024-01""",3297,1069903,325.0,7.0
"""2024-02""",3230,1051770,326.0,6.0
"""2024-03""",6672,2191528,328.0,13.0
"""2024-04""",3238,1042062,322.0,6.0
"""2024-05""",5068,1643682,324.0,10.0
…,…,…,…,…
"""2024-08""",3421,1115879,326.0,7.0
"""2024-09""",3262,1064738,326.0,7.0
"""2024-10""",5709,1844991,323.0,11.0


In [28]:
monthly_festival_sales

year_month,festival,transactions,total_revenue
str,str,u32,i64
"""2024-01""",null,3297,1069903
"""2024-02""",null,3230,1051770
"""2024-03""","""Holi""",3331,1097119
"""2024-03""",null,3341,1094409
"""2024-04""",null,3238,1042062
…,…,…,…
"""2024-10""",null,3375,1091575
"""2024-10""","""Diwali""",2334,753416
"""2024-11""",null,3285,1059115


In [29]:
sales_df

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,year_month,month_name
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,str,str
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,"""2024-01""","""January"""
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,"""2024-01""","""January"""
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,"""2024-01""","""January"""
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,"""2024-01""","""January"""
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,"""2024-01""","""January"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""T48069""","""c243""","""p3""",2024-12-31 00:00:00,null,"""Solapur""","""Lip Balm""","""Makeup""","""Eiravati Pillai""",44,"""Male""",149,149,"""2024-12""","""December"""
"""T48763""","""c716""","""p0""",2024-12-31 00:00:00,null,"""Parbhani""","""Aloe Vera Gel""","""Skincare""","""Falguni Bakshi""",19,"""Female""",299,299,"""2024-12""","""December"""
"""T48997""","""c609""","""p2""",2024-12-31 00:00:00,null,"""Ambala""","""Herbal Shampoo""","""Haircare""","""Samarth Ray""",35,"""Female""",199,199,"""2024-12""","""December"""


# 10_day10_customer_segmentation.ipynb
==================================================

## Imports


In [1]:
import polars as pl
from pathlib import Path

# Data Path


In [2]:
data_folder = Path("../data")

# Load Dataset


In [3]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Customer Metrics


In [5]:
customer_metrics = sales_df.group_by("customer_id").agg(
    pl.count().alias("total_orders"),
    pl.col("revenue").sum().alias("total_spent"),
    pl.col("revenue").mean().alias("avg_order_value"),
    pl.col("purchase_date").min().alias("first_purchase"),
    pl.col("purchase_date").max().alias("last_purchase")
)

customer_metrics

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_18920\363853550.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_orders"),


customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
str,u32,i64,f64,datetime[μs],datetime[μs]
"""c721""",48,15702,327.125,2024-01-12 00:00:00,2024-12-14 00:00:00
"""c277""",74,23726,320.621622,2024-01-10 00:00:00,2024-12-24 00:00:00
"""c663""",32,10118,316.1875,2024-01-04 00:00:00,2024-12-30 00:00:00
"""c618""",37,11813,319.27027,2024-01-01 00:00:00,2024-12-26 00:00:00
"""c272""",64,19886,310.71875,2024-01-01 00:00:00,2024-12-27 00:00:00
…,…,…,…,…,…
"""c195""",53,17197,324.471698,2024-02-05 00:00:00,2024-12-30 00:00:00
"""c962""",64,21286,332.59375,2024-01-02 00:00:00,2024-12-18 00:00:00
"""c335""",40,13560,339.0,2024-01-06 00:00:00,2024-12-25 00:00:00


## Customer Lifetime


In [ ]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("last_purchase") - pl.col("first_purchase")
    ).dt.total_days().alias("customer_lifetime_days")
)

# Purchase Frequency


In [7]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days") + 1)
    ).alias("purchase_frequency")
)

## Customer Segments


In [ ]:
customer_metrics = customer_metrics.with_columns(
    pl.when(pl.col("total_spent") > 20000)
    .then(pl.lit("VIP"))

    .when(pl.col("total_spent") > 10000)
    .then(pl.lit("Loyal"))

    .when(pl.col("total_spent") > 5000)
    .then(pl.lit("Regular"))

    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")
)

# Segment Distribution


In [10]:
segment_distribution = customer_metrics.group_by("customer_type").agg(
    pl.count().alias("customers"),
    pl.col("total_spent").sum().alias("segment_revenue"),
).sort("segment_revenue", descending=True)

# Adding revenue_share_pct column

segment_distribution = segment_distribution.with_columns(

    (
        pl.col("segment_revenue")/pl.col("segment_revenue").sum() * 100

    ).round().alias("revenue_share_pct")

)

segment_distribution

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_18920\2154365367.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("customers"),


customer_type,customers,segment_revenue,revenue_share_pct
str,u32,i64,f64
"""Loyal""",629,9589136,59.0
"""VIP""",232,5612876,35.0
"""Regular""",117,953773,6.0
"""Low Value""",22,90165,1.0


Save the dataset


In [11]:
customer_metrics.write_parquet(data_folder/"customer_metrics.parquet")

# 11_day11_duckdb_analysis.ipynb
==================================================

# Imports


In [40]:
import duckdb
import polars as pl
from pathlib import Path

# Data Path


In [41]:
data_folder = Path("../data")

# Connect to DuckDB


In [42]:
con = duckdb.connect()

## Query Parquet File


In [43]:
sales = con.execute(f"""
    select * from read_parquet('{data_folder}/sales_cleaned.parquet')
    limit 5
""").fetchdf()

sales

,transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
0,T82,c377,p3,2024-01-01,None,Jorhat,Lip Balm,Makeup,Harshil Varty,21,Female,149,149
1,T910,c202,p1,2024-01-01,None,Indore,Vitamin C Serum,Skincare,Tanvi Chana,51,Male,499,499
2,T1273,c752,p2,2024-01-01,None,Secunderabad,Herbal Shampoo,Haircare,Idika Roy,48,Male,199,199
3,T1369,c739,p0,2024-01-01,None,Bhatpara,Aloe Vera Gel,Skincare,Harita Comar,49,Female,299,299
4,T2858,c169,p1,2024-01-01,None,Bhagalpur,Vitamin C Serum,Skincare,Zayyan Mander,51,Female,499,499


# SQL Revenue by Category


In [44]:
category_sales = con.execute(f"""
    
    select category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by category
    order by total_revenue desc                         

    """).fetchdf()

print(category_sales)

   category  transactions  total_revenue  avg_order_value
0  Skincare         30306     12317344.0            406.0
1  Haircare          9789      2191061.0            224.0
2    Makeup          9905      1737545.0            175.0


# SQL Monthly Revenue


In [45]:
monthly_sale_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month
    order by month

    """).fetchdf()

print(monthly_sale_sql)

      month  transactions  total_revenue
0   2024-01          3297      1069903.0
1   2024-02          3230      1051770.0
2   2024-03          6672      2191528.0
3   2024-04          3238      1042062.0
4   2024-05          5068      1643682.0
5   2024-06          4970      1617130.0
6   2024-07          3379      1098971.0
7   2024-08          3421      1115879.0
8   2024-09          3262      1064738.0
9   2024-10          5709      1844991.0
10  2024-11          4315      1394035.0
11  2024-12          3439      1111261.0


# Customer Segmentation SQL Table


In [46]:
customer_metrics_sql = con.execute(f"""

    select customer_id,
    count(*) as total_orders,
    sum(revenue) as total_spent,
    avg(revenue) as avg_order_value,
    min(purchase_date) as first_purchase,
    max(purchase_date) as last_purchase,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by customer_id

    """).fetchdf()

customer_metrics_sql.head()

,customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
0,c377,27,8923.0,330.481481,2024-01-01,2024-12-25
1,c179,53,16747.0,315.981132,2024-01-01,2024-12-30
2,c394,37,12313.0,332.783784,2024-01-01,2024-12-21
3,c675,63,18937.0,300.587302,2024-01-01,2024-12-25
4,c178,74,23476.0,317.243243,2024-01-01,2024-12-24


# City Performance SQL Table


In [47]:
city_performance_sql = con.execute(f"""

    select city,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(sum(revenue) * 100.0 /sum(sum(revenue)) over(),2) as revenue_share_pct
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by city
    order by total_revenue desc                        

    """).fetchdf()

print(city_performance_sql)

             city  transactions  total_revenue  revenue_share_pct
0    South Dumdum           446       144004.0               0.89
1      Berhampore           421       138479.0               0.85
2       Allahabad           435       137415.0               0.85
3       Ghaziabad           406       133544.0               0.82
4          Haldia           411       133489.0               0.82
..            ...           ...            ...                ...
296       Lucknow            31         8619.0               0.05
297        Tumkur            26         8224.0               0.05
298       Raiganj            16         5634.0               0.03
299         Morbi            15         4735.0               0.03
300       Panipat            13         4037.0               0.02

[301 rows x 4 columns]


# Festival & Monthly Analysis


In [48]:
monthly_festival_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    festival,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month,festival
    order by month

    """).fetchdf()

print(monthly_festival_sql)

      month festival  transactions  total_revenue
0   2024-01     None          3297      1069903.0
1   2024-02     None          3230      1051770.0
2   2024-03     Holi          3331      1097119.0
3   2024-03     None          3341      1094409.0
4   2024-04     None          3238      1042062.0
5   2024-05   Summer          1617       530483.0
6   2024-05     None          3451      1113199.0
7   2024-06   Summer          1601       520999.0
8   2024-06     None          3369      1096131.0
9   2024-07     None          3379      1098971.0
10  2024-08     None          3421      1115879.0
11  2024-09     None          3262      1064738.0
12  2024-10   Diwali          2334       753416.0
13  2024-10     None          3375      1091575.0
14  2024-11     None          3285      1059115.0
15  2024-11   Diwali          1030       334920.0
16  2024-12     None          3439      1111261.0


# Product performance table


In [49]:

product_perf_sql = con.execute(f"""

    select product_name,
    category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by product_name,category
    order by total_revenue desc                         

    """).fetchdf()

print(product_perf_sql)

      product_name  category  transactions  total_revenue  avg_order_value
0  Sunscreen SPF50  Skincare          5056      3028544.0            599.0
1  Vitamin C Serum  Skincare          4943      2466557.0            499.0
2        Face Wash  Skincare          5015      2000985.0            399.0
3      Body Lotion  Skincare          4979      1737671.0            349.0
4        Face Pack  Skincare          5285      1580215.0            299.0
5    Aloe Vera Gel  Skincare          5028      1503372.0            299.0
6         Hair Oil  Haircare          4861      1210389.0            249.0
7      Nail Polish    Makeup          5234      1041566.0            199.0
8   Herbal Shampoo  Haircare          4928       980672.0            199.0
9         Lip Balm    Makeup          4671       695979.0            149.0


# Save BI-ready tables to Parquet


In [ ]:
customer_metrics_sql.to_parquet(
    data_folder / "customer_metrics_duckdb.parquet")
city_performance_sql.to_parquet(
    data_folder / "city_performance_duckdb.parquet")
monthly_festival_sql.to_parquet(
    data_folder / "monthly_festival_duckdb.parquet")
product_perf_sql.to_parquet(data_folder / "products_perf_duckdb.parquet")

## Verify the Parquet files


In [52]:
print("Customer metrics rows:", len(customer_metrics_sql))
print("City performance rows:", len(city_performance_sql))
print("Monthly festival rows:", len(monthly_festival_sql))
print("Product performance rows:", len(product_perf_sql))

Customer metrics rows: 1000
City performance rows: 301
Monthly festival rows: 17
Product performance rows: 10
